In [4]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [5]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location'
tif_list = glob(tif_list + '/*[!xlsx|ipynb|ad]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193345',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193346',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193347',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP10193348',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759311',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759312',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP8759313',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258463',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258464',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA_LngSP9258467',
 '/data1/r20user3/shared_project/Hist2Cell/data/human_lung_cell2location/WSA

In [16]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['WSA_LngSP10193345',
 'WSA_LngSP10193346',
 'WSA_LngSP10193347',
 'WSA_LngSP10193348',
 'WSA_LngSP8759311',
 'WSA_LngSP8759312',
 'WSA_LngSP8759313',
 'WSA_LngSP9258463',
 'WSA_LngSP9258464',
 'WSA_LngSP9258467',
 'WSA_LngSP9258468']

In [18]:
# prepare my GPU

gpu_list = [3]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [21]:
# # prepare the feature extractor model - KimiaNet

# model = models.densenet121(pretrained=True)
# model.features = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(output_size=(1, 1)))
# num_ftrs = model.classifier.in_features
# model_final = fully_connected(model.features, num_ftrs, 250+39)

# # KimiaNetPyTorchWeights_path = "/home/r10user3/Documents/Human-breast-cancer-in-situ-capturing-transcriptomics/KimiaNetPyTorchWeights.pth"
# # Checkpoint_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/densenet_multi_cell_gene_epoch500_best_cell_average.pth'

# # checkpoint = torch.load(KimiaNetPyTorchWeights_path)
# # checkpoint = torch.load(Checkpoint_path)

# new_state_dict = collections.OrderedDict()
# for k, v in checkpoint.items():
#     if 'fc_4' in k:
#         continue
#     name = k[7:]  # remove "module."
#     new_state_dict[name] = v
# model2_dict = model_final.state_dict()
# state_dict = {k:v for k,v in new_state_dict.items() if k in model2_dict.keys()}
# model2_dict.update(state_dict)
# model_final.load_state_dict(model2_dict)
# model = model_final
# model = model.to(device)
# model

In [22]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [23]:
batch_size = 512
patch_dir = "/home/r20user3/Documents/Project/Hist2Cell/data/human_lung_cell2location"

graphs = dict()
test_data_list = []

adata = sc.read("/home/r20user3/Documents/Project/Hist2Cell/data/human_lung_cell2location/sp.X_norm5e4_log1p.h5ad")
center_col = adata.obs.array_col
center_row = adata.obs.array_row

for slide in test_slides:
    print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=8)
    
    spots_coord = pd.read_csv(os.path.join("/home/r20user3/Documents/Project/Hist2Cell/data/human_lung_cell2location", slide, "spots.csv"), index_col=0)

    with torch.no_grad():
        model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []
        
        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            
            test_prediction_array.append(data.cpu().detach().numpy())
            
            # break


    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    test_names = list()
    for names in test_name_array:
        test_names=test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(center_col[item]), int(center_row[item])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 3 or x2 >= x1 + 3 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")
    # break

WSA_LngSP10193345
OK
WSA_LngSP10193346
OK
WSA_LngSP10193347
OK
WSA_LngSP10193348
OK
WSA_LngSP8759311
OK
WSA_LngSP8759312
OK
WSA_LngSP8759313
OK
WSA_LngSP9258463
OK
WSA_LngSP9258464
OK
WSA_LngSP9258467
OK
WSA_LngSP9258468
OK


In [24]:
graphs['WSA_LngSP10193345']['features'].shape

(3234, 3, 224, 224)

In [25]:
graphs['WSA_LngSP10193345']['coord'].shape

(3234, 2)

In [27]:
graphs['WSA_LngSP10193345']['names']

['WSA_LngSP10193345_GGATTGAAGTAGCCTC-1',
 'WSA_LngSP10193345_CAGAGTGATTTAACGT-1',
 'WSA_LngSP10193345_TTCACTTCCTAGAACG-1',
 'WSA_LngSP10193345_AGCGGACACTTCGTAG-1',
 'WSA_LngSP10193345_TTGACCAGGAACAACT-1',
 'WSA_LngSP10193345_CTCGCATTGCATAGCC-1',
 'WSA_LngSP10193345_TGCCACCTGGCGAAAC-1',
 'WSA_LngSP10193345_TTCGCTATCTGACGTG-1',
 'WSA_LngSP10193345_ATACGGGTTTCGATTG-1',
 'WSA_LngSP10193345_CTTACACGGTATTCCA-1',
 'WSA_LngSP10193345_GGGTCCTTGGAAGAAG-1',
 'WSA_LngSP10193345_TCTATCGGTCGCAACA-1',
 'WSA_LngSP10193345_CGGAGCAATTTAATCG-1',
 'WSA_LngSP10193345_CTTTGGCAGACAGAGT-1',
 'WSA_LngSP10193345_CTCGGTACCACTGCTC-1',
 'WSA_LngSP10193345_TACATGCCGGAATTGT-1',
 'WSA_LngSP10193345_TGCTCGGCGAAACCCA-1',
 'WSA_LngSP10193345_GATCCCTTTATACTGC-1',
 'WSA_LngSP10193345_CAATTTCGTATAAGGG-1',
 'WSA_LngSP10193345_AATACCTGATGTGAAC-1',
 'WSA_LngSP10193345_CCCTTTGACAGGTCTT-1',
 'WSA_LngSP10193345_CGTAACGGAACGATCA-1',
 'WSA_LngSP10193345_AAACTCGGTTCGCAAT-1',
 'WSA_LngSP10193345_TAAGTAACATCTTGAC-1',
 'WSA_LngSP10193

In [28]:
graph = graphs
for slide in graph:
    print(slide)

WSA_LngSP10193345
WSA_LngSP10193346
WSA_LngSP10193347
WSA_LngSP10193348
WSA_LngSP8759311
WSA_LngSP8759312
WSA_LngSP8759313
WSA_LngSP9258463
WSA_LngSP9258464
WSA_LngSP9258467
WSA_LngSP9258468


In [29]:
from torch import Tensor
import torch
import os

for slide_name in graph:
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    y = Tensor(graph[slide_name]['labels'])
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    
    torch.save(data, os.path.join("/home/r20user3/Documents/Project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location", slide_name+".pt"))

In [1]:
import torch

data1 = torch.load("/home/r20user3/Documents/Project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location/WSA_LngSP8759311.pt")
data2 = torch.load("/home/r20user3/Documents/Project/Hist2Cell/code/data_preprocessing/rawimg_graph/human_lung_cell2location/WSA_LngSP8759312.pt")

In [2]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
data

DataBatch(x=[4815, 3, 224, 224], edge_index=[2, 32249], y=[4815, 330], pos=[4815, 2], batch=[4815], ptr=[3])

In [19]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

for sampled_data in loader:
    break
len(sampled_data.n_id)

19